# 06 · Teste Visual de Cards — Teams

**Objetivo:** disparar e validar cards no Microsoft Teams via Power Automate.

| Etapa | Descrição |
|---|---|
| A | Carregar dados (CSVs já gerados) |
| B | Mini-backtest: coletar o primeiro evento de cada tipo |
| C | Preview dos payloads (JSON completo) |
| D | Disparar no SharePoint com prefixo `[TESTE]` (dados sintéticos) |
| E | Limpar itens `[TESTE]` do SP |
| **F** | **Disparo real: engine no ciclo atual → SP sem prefixo** |

In [1]:
# ── Configuração ──────────────────────────────────────────────────────────────
MAQUINA    = 'FB14'
LIST_NAME  = 'Gatilhos_Selagem'

# Prefixo no campo Title para identificar itens de teste no SP
PREFIXO    = '[TESTE]'

# Tipos a disparar — pode remover tipos se quiser testar só alguns
TIPOS_ALVO = ['RISCO', 'CRITICO', 'EMERGENCIAL', 'RED', 'AMARELO', 'REVISAO']
# ─────────────────────────────────────────────────────────────────────────────

## Etapa A — Carregar dados

In [2]:
import sys, json, tempfile, warnings
from pathlib import Path
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd

from src.trigger_engine import TriggerEngine
from src.predictor     import load_troca_dates
from src.sku_normalizer import normalizar_media

_NB_DIR = Path.cwd()
_ROOT   = _NB_DIR.parent

df_raw = pd.read_csv('00_hour_prev.csv', parse_dates=['Timestamp'])
df_raw['Timestamp'] = pd.to_datetime(df_raw['Timestamp'], utc=True)
df_raw = df_raw.set_index('Timestamp').sort_index()

df_sku = pd.read_csv('sku_dates.csv', parse_dates=['index'])
df_sku['ts'] = pd.to_datetime(df_sku['index'], utc=True)

df_hourly = normalizar_media(df_raw, df_sku, col_sku='bag1_sku', col_media='Media')
cobertura = df_hourly['Media_norm'].notna().mean()
media_col = 'Media_norm' if cobertura >= 0.50 else 'Media'

troca_dates = load_troca_dates('troca_modulo.csv')

print(f'Dados    : {len(df_hourly):,} leituras ({df_hourly.index.min().date()} → {df_hourly.index.max().date()})')
print(f'Trocas   : {len(troca_dates)}  (última: {troca_dates[-1].date()})')
print(f'Media_col: {media_col!r}  (cobertura SKU: {cobertura:.0%})')

Dados    : 3,255 leituras (2022-06-27 → 2026-05-25)
Trocas   : 15  (última: 2026-05-19)
Media_col: 'Media_norm'  (cobertura SKU: 100%)


## Etapa B — Mini-backtest: coletar primeiro evento de cada tipo

Percorre os ciclos históricos do mais antigo para o mais recente e para
assim que tiver um exemplo de cada tipo em `TIPOS_ALVO`.

In [3]:
exemplos  = {}           # gatilho → (day, TriggerEvent)
pendentes = set(TIPOS_ALVO)

for i, (t_ini, t_fim) in enumerate(zip(troca_dates[:-1], troca_dates[1:])):
    if not pendentes:
        break

    ciclo_df = df_hourly[(df_hourly.index >= t_ini) & (df_hourly.index < t_fim)]
    if ciclo_df.empty:
        continue

    _state_tmp = Path(tempfile.mktemp(suffix='.json'))
    _eng       = TriggerEngine(MAQUINA, _state_tmp)
    _dias      = pd.DatetimeIndex(sorted({ts.normalize() for ts in ciclo_df.index}))

    for _day in _dias:
        if not pendentes:
            break
        _df_ate = df_hourly[df_hourly.index <= _day + pd.Timedelta(hours=23, minutes=59)]
        try:
            evs = _eng.evaluate(
                _df_ate,
                troca_date    = t_ini,
                sp_client     = None,
                today         = _day,
                troca_dates   = troca_dates,
                media_col     = media_col,
            )
            for ev in evs:
                if ev.gatilho in pendentes:
                    exemplos[ev.gatilho] = (_day, ev)
                    pendentes.discard(ev.gatilho)
                    print(f'  ✓ {ev.gatilho:<14} encontrado — ciclo {i}  {_day.date()}  dia {ev.idade_maintacker}')
        except Exception as exc:
            print(f'  [ERRO] ciclo {i} {_day.date()}: {exc}')

    if _state_tmp.exists():
        _state_tmp.unlink()

print()
print(f'Coletados : {sorted(exemplos.keys())}')
if pendentes:
    print(f'Não encontrados: {sorted(pendentes)}')

[dry-run] RISCO não persistido (sp_client=None).
[dry-run] RISCO não persistido (sp_client=None).
[dry-run] REVISAO não persistido (sp_client=None).
[dry-run] AMARELO não persistido (sp_client=None).
[dry-run] RISCO não persistido (sp_client=None).
[dry-run] CRITICO não persistido (sp_client=None).
[dry-run] REVISAO não persistido (sp_client=None).
[dry-run] CRITICO não persistido (sp_client=None).
[dry-run] EMERGENCIAL não persistido (sp_client=None).
[dry-run] RED não persistido (sp_client=None).


  ✓ RISCO          encontrado — ciclo 1  2025-03-12  dia 12
  ✓ REVISAO        encontrado — ciclo 1  2025-03-20  dia 20
  ✓ AMARELO        encontrado — ciclo 1  2025-03-22  dia 22
  ✓ CRITICO        encontrado — ciclo 1  2025-03-25  dia 25
  ✓ EMERGENCIAL    encontrado — ciclo 1  2025-03-31  dia 31
  ✓ RED            encontrado — ciclo 1  2025-03-31  dia 31

Coletados : ['AMARELO', 'CRITICO', 'EMERGENCIAL', 'RED', 'REVISAO', 'RISCO']


## Etapa C — Preview dos payloads

Mostra o JSON completo que vai para o campo `TeamsPayload` de cada tipo.

In [4]:
for tipo in TIPOS_ALVO:
    if tipo not in exemplos:
        print(f'  {tipo}: não encontrado\n')
        continue

    day, ev = exemplos[tipo]
    print(f'\n{"═"*65}')
    print(f'  {tipo}  |  {day.date()}  |  dia {ev.idade_maintacker}  |  [{ev.severidade}]')
    print(f'{"═"*65}')

    if ev.teams_payload:
        try:
            print(json.dumps(json.loads(ev.teams_payload), indent=2, ensure_ascii=False))
        except Exception:
            print(ev.teams_payload)
    else:
        print('  (sem payload)')


═════════════════════════════════════════════════════════════════
  RISCO  |  2025-03-12  |  dia 12  |  [INFO]
═════════════════════════════════════════════════════════════════
{
  "type": "AdaptiveCard",
  "$schema": "http://adaptivecards.io/schemas/adaptive-card.json",
  "version": "1.4",
  "body": [
    {
      "type": "Container",
      "style": "warning",
      "bleed": true,
      "items": [
        {
          "type": "ColumnSet",
          "columns": [
            {
              "type": "Column",
              "width": "stretch",
              "items": [
                {
                  "type": "TextBlock",
                  "text": "⚠️ RISCO — Leitura Anômala",
                  "weight": "Bolder",
                  "size": "Large",
                  "color": "Light",
                  "wrap": true
                },
                {
                  "type": "TextBlock",
                  "text": "Força de Selagem — Rolo Maintacker",
                  "color": "Light",


## Etapa D — Disparar no SharePoint → Teams

Cada item criado tem `Title` com prefixo `[TESTE]`.  
O Power Automate vai pegar esses itens e postar no canal do Teams — **é assim que veremos o card real**.

In [5]:
from dotenv import dotenv_values
from src.sharepoint_methods import SharePointClient

creds = dotenv_values(_ROOT / 'sharepoint.ev')
sp    = SharePointClient(
    'https://kimberlyclark.sharepoint.com/Sites/H945',
    creds['SP_USER'],
    creds['SP_PASS'],
)
print('SharePoint: conectado.\n')

itens_sp = []
for tipo in TIPOS_ALVO:
    if tipo not in exemplos:
        print(f'  {tipo}: pulado (não encontrado na Etapa B)')
        continue

    day, ev = exemplos[tipo]
    title   = f"{PREFIXO} {MAQUINA} | {tipo} | {day.date()}"
    itens_sp.append({
        'Title'       : title,
        'Maquina'     : MAQUINA,
        'TeamsPayload': ev.teams_payload,
    })
    print(f'  {tipo:<14}  →  "{title}"')

print(f'\nDisparando {len(itens_sp)} item(s) para "{LIST_NAME}"...')
ids = sp.insert_list_item(LIST_NAME, itens_sp)

print()
for item, sp_id in zip(itens_sp, ids):
    status = f'ID={sp_id}' if sp_id else 'FALHOU'
    print(f'  [{status}]  {item["Title"]}')

ids_criados = [x for x in ids if x]
print(f'\n✓ {len(ids_criados)} / {len(itens_sp)} inseridos')
print('  Power Automate vai postar no Teams em instantes.')

SharePoint: conectado.

  RISCO           →  "[TESTE] FB14 | RISCO | 2025-03-12"
  CRITICO         →  "[TESTE] FB14 | CRITICO | 2025-03-25"
  EMERGENCIAL     →  "[TESTE] FB14 | EMERGENCIAL | 2025-03-31"
  RED             →  "[TESTE] FB14 | RED | 2025-03-31"
  AMARELO         →  "[TESTE] FB14 | AMARELO | 2025-03-22"
  REVISAO         →  "[TESTE] FB14 | REVISAO | 2025-03-20"

Disparando 6 item(s) para "Gatilhos_Selagem"...
6 items created

  [ID=124]  [TESTE] FB14 | RISCO | 2025-03-12
  [ID=125]  [TESTE] FB14 | CRITICO | 2025-03-25
  [ID=126]  [TESTE] FB14 | EMERGENCIAL | 2025-03-31
  [ID=127]  [TESTE] FB14 | RED | 2025-03-31
  [ID=128]  [TESTE] FB14 | AMARELO | 2025-03-22
  [ID=129]  [TESTE] FB14 | REVISAO | 2025-03-20

✓ 6 / 6 inseridos
  Power Automate vai postar no Teams em instantes.


## Etapa E — Limpar itens de teste (opcional)

Executa somente quando `LIMPAR = True`.  
Remove todos os itens cujo `Title` começa com o prefixo de teste.

In [6]:
LIMPAR = True   # ← mudar para True para deletar os itens de teste

if not LIMPAR:
    print('LIMPAR=False — nada removido.')
else:
    df_sp    = sp.query_large_list(LIST_NAME)
    mask     = df_sp['Title'].str.startswith(PREFIXO, na=False)
    ids_test = df_sp.loc[mask, 'ID'].astype(int).tolist()

    if not ids_test:
        print(f'Nenhum item com prefixo "{PREFIXO}" encontrado.')
    else:
        print(f'Removendo {len(ids_test)} item(s) de teste: {ids_test}')
        sp.delete_list_items(LIST_NAME, ids_test)
        print('✓ Removidos.')

LIMPAR=False — nada removido.


## Etapa F — Disparo real: engine no ciclo atual

Usa `state_fb14.json` real e posta no SharePoint **sem prefixo** —
comportamento idêntico ao pipeline de produção.

> **Atenção:** atualiza cooldowns no state real. Se nenhum gatilho disparar,
> verifique cooldowns ativos em `state_fb14.json`.

In [ ]:
from dotenv import dotenv_values
from src.sharepoint_methods import SharePointClient

# ── Conexão SP (reusa se Etapa D já rodou) ────────────────────────────────────
if 'sp' not in vars() or sp is None:
    _creds = dotenv_values(_ROOT / 'sharepoint.ev')
    sp = SharePointClient(
        _creds.get('SP_URL', 'https://kimberlyclark.sharepoint.com/Sites/H945'),
        _creds['SP_USER'],
        _creds['SP_PASS'],
    )
    print('SharePoint: conectado.')

# ── Engine com state real ──────────────────────────────────────────────────────
_state_real = _ROOT / 'state_fb14.json'
engine_real = TriggerEngine(MAQUINA, _state_real)

ultima_troca = troca_dates[-1]
hoje         = df_hourly.index.max().normalize()

print(f'Última troca : {ultima_troca.date()}')
print(f'Avaliando em : {hoje.date()}')
print()

eventos = engine_real.evaluate(
    df_hourly,
    troca_date  = ultima_troca,
    sp_client   = sp,
    list_name   = LIST_NAME,
    troca_dates = troca_dates,
    media_col   = media_col,
)

if not eventos:
    print('Nenhum gatilho disparado (cooldowns ativos ou thresholds não atingidos).')
else:
    for ev in eventos:
        status = f'SP ID={ev.sp_item_id}' if ev.sp_item_id else 'não persistido'
        print(f'  ✓ {ev.gatilho:<12} [{ev.severidade}]  dia {ev.idade_maintacker}  p_risk={ev.p_risk:.3f}  → {status}')